# 07 — Model Export: ONNX / TorchScript (Kaggle / Local)

Exports trained models cho optimized local inference:
- **ONNX** — cross-platform inference (CPU/GPU)
- **TorchScript** — mobile/embedded deployment

**Kaggle**: Export vào `/kaggle/working/storage/exports/` → Save Version để persist.

In [ ]:
# ── Cell 0: Environment + Paths ───────────────────────────────────────────────
import sys, os
from pathlib import Path

if os.path.exists('/kaggle/working/project/training/notebooks'):
    sys.path.insert(0, '/kaggle/working/project/training/notebooks')

from _env import *

os.makedirs(os.path.join(EXPORTS_DIR, 'onnx'), exist_ok=True)
os.makedirs(os.path.join(EXPORTS_DIR, 'torchscript'), exist_ok=True)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'ENV         : {ENV_NAME}')
print(f'Device      : {device}')
print(f'WEIGHTS_DIR : {WEIGHTS_DIR}')
print(f'EXPORTS_DIR : {EXPORTS_DIR}')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Nhớ Save Version sau khi export!')

In [ ]:
# ── Cell 1: Cài dependencies ────────────────────────────────────────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'onnx', 'onnxruntime', 'optimum'], check=False)
print('Deps installed')

In [ ]:
# ── Cell 2: Export Dialect Classifier → TorchScript + ONNX ───────────────────────
import json

DIALECT_MODEL_PATH = os.path.join(WEIGHTS_DIR, 'dialect', 'dialect_model.pt')

if os.path.exists(DIALECT_MODEL_PATH):
    from transformers import WhisperModel
    import torch.nn as nn

    checkpoint = torch.load(DIALECT_MODEL_PATH, map_location='cpu')
    n_province  = len(checkpoint['province2id'])
    n_region    = len(checkpoint['region2id'])
    n_ethnicity = len(checkpoint['ethnicity2id'])
    backbone    = checkpoint.get('backbone', 'vinai/PhoWhisper-base')

    class DialectClassifier(nn.Module):
        def __init__(self, backbone_name, n_prov, n_reg, n_eth):
            super().__init__()
            whisper = WhisperModel.from_pretrained(backbone_name)
            self.encoder = whisper.encoder
            hidden = whisper.config.d_model
            self.dropout = nn.Dropout(0.1)
            self.province_head  = nn.Linear(hidden, n_prov)
            self.region_head    = nn.Linear(hidden, n_reg)
            self.ethnicity_head = nn.Linear(hidden, n_eth)

        def forward(self, input_features):
            hidden = self.encoder(input_features).last_hidden_state.mean(dim=1)
            hidden = self.dropout(hidden)
            return self.province_head(hidden), self.region_head(hidden), self.ethnicity_head(hidden)

    model = DialectClassifier(backbone, n_province, n_region, n_ethnicity)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    dummy_input = torch.randn(1, 80, 3000)

    # TorchScript
    scripted = torch.jit.trace(model, dummy_input)
    ts_path  = os.path.join(EXPORTS_DIR, 'torchscript', 'dialect_classifier.pt')
    scripted.save(ts_path)
    print(f'✓ Dialect → TorchScript: {ts_path}')

    # ONNX
    onnx_path = os.path.join(EXPORTS_DIR, 'onnx', 'dialect_classifier.onnx')
    torch.onnx.export(
        model, dummy_input, onnx_path,
        input_names=['input_features'],
        output_names=['province_logits', 'region_logits', 'ethnicity_logits'],
        dynamic_axes={'input_features': {0: 'batch', 2: 'frames'}},
        opset_version=17,
    )
    print(f'✓ Dialect → ONNX: {onnx_path}')
else:
    print(f'Dialect model không tìm thấy: {DIALECT_MODEL_PATH}')
    print('Chạy 02_dialect_identification.ipynb trước.')

In [ ]:
# ── Cell 3: Export ASR Wav2Vec2 → ONNX via Optimum ─────────────────────────────
if os.path.exists(ASR_WEIGHT_DIR) and os.path.exists(os.path.join(ASR_WEIGHT_DIR, 'config.json')):
    from optimum.onnxruntime import ORTModelForCTC
    from transformers import Wav2Vec2Processor

    print('Exporting Wav2Vec2 → ONNX via Optimum...')
    ort_model = ORTModelForCTC.from_pretrained(
        ASR_WEIGHT_DIR, export=True,
        provider='CUDAExecutionProvider' if device == 'cuda' else 'CPUExecutionProvider',
    )
    wav2vec2_onnx_dir = os.path.join(EXPORTS_DIR, 'onnx', 'wav2vec2')
    os.makedirs(wav2vec2_onnx_dir, exist_ok=True)
    ort_model.save_pretrained(wav2vec2_onnx_dir)

    Wav2Vec2Processor.from_pretrained(ASR_WEIGHT_DIR).save_pretrained(wav2vec2_onnx_dir)
    print(f'✓ Wav2Vec2 ONNX → {wav2vec2_onnx_dir}')
else:
    print(f'ASR model không tìm thấy tại {ASR_WEIGHT_DIR}')
    print('Chạy 03_asr_wav2vec2_new.ipynb trước.')

In [ ]:
# ── Cell 4: Verify ONNX inference ───────────────────────────────────────────────
import onnxruntime as ort
import numpy as np

onnx_path = os.path.join(EXPORTS_DIR, 'onnx', 'dialect_classifier.onnx')
if os.path.exists(onnx_path):
    sess    = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    dummy   = np.random.randn(1, 80, 3000).astype(np.float32)
    outputs = sess.run(None, {'input_features': dummy})
    print(f'✓ ONNX inference verified:')
    print(f'  Province logits : {outputs[0].shape}')
    print(f'  Region logits   : {outputs[1].shape}')
    print(f'  Ethnicity logits: {outputs[2].shape}')
else:
    print('ONNX model không tìm thấy, bỏ qua verification')

In [ ]:
# ── Cell 5: Tóm tắt files đã export ──────────────────────────────────────────────
print('\nExported model files:')
for root, dirs, files in os.walk(EXPORTS_DIR):
    level    = root.replace(EXPORTS_DIR, '').count(os.sep)
    indent   = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        fpath   = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'{indent}  {f} ({size_mb:.1f} MB)')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Nhấn Save Version để persist exports!')

## ✓ Model Export Complete

| Platform | Export location |
|----------|-----------------|
| Kaggle | `/kaggle/working/storage/exports/` |
| Colab | `Drive/GenAI_ViMD/storage/exports/` |

### Chạy inference local

1. Download output từ Kaggle → local
2. Set `.env`: `ASR_MODEL_PATH=storage/exports/onnx/wav2vec2`
3. Khởi động backend: `python -m backend.main`